In [ ]:
from pathlib import Path
import numpy as np
from sklearn.preprocessing import StandardScaler, Normalizer

import tensorflow as tf
from eval_core import evaluate
from log_core import log_run

import importlib, log_core
importlib.reload(log_core)
from log_core import log_run

#%run helpers.ipynb


#HIPERPARAMETRY
latent_dim    = 4
depth         = 2        
width_first   = 256
beta_kl       = 0.01

batch_size    = 32         
dropout       = 0.05
l2            = 0
learning_rate = 1e-3  

# === 1. Ładowanie danych ===
root    = Path(r"E:/marcel.furs/Documents/Data_Run_Walk/NajnowszedoMgr")
X_train = np.load(root / "dataset_train_multi_fixed_OSTATECZNY.npy")    # (N_train, 100, 55, 3)
X_test  = np.load(root / "dataset_test_multi_fixed_OSTATECZNY.npy")     # (N_test,  100, 55, 3)
X_anom  = np.load(root / "dataset_anomaly_multi_fixed_OSTATECZNY.npy")  # (N_anom,  100, 55, 3)

# === 2. Flatten → (N, 16500) ===
def flatten_seqs(X):
    # 100 klatek × 55 markerów × 3 osie = 100*165 = 16500
    N, T, M, C = X.shape
    return X.reshape(N, T * M * C)

X_train = flatten_seqs(X_train)
X_test  = flatten_seqs(X_test)
X_anom  = flatten_seqs(X_anom)

# === 3. Z-score per-feature (fit tylko na X_train) ===
#normalizer = Normalizer() od 0 do 1
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
X_anom  = scaler.transform(X_anom)


# === 4. Sprawdź y ===
print("Shapes:", X_train.shape, X_test.shape, X_anom.shape)
print("Min/max po standaryzacji:", X_train.min(), X_train.max())

In [ ]:
# SETUP

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import backend as K

In [ ]:
# Create a sampling layer

In [ ]:
class Sampling(layers.Layer):
    """Reparameterization trick: z = μ + ε·σ,  ε∼N(0,I)."""
    
    def call(self, inputs):
        z_mean, z_logvar = inputs
        epsilon = tf.random.normal(tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_logvar) * epsilon

In [ ]:
# Build the encoder (shape=100, 55, 3)100 klatek 55 markerów 3 współrzędne XYZ

In [ ]:
input_dim = X_train.shape[1]  # 16500 

encoder_inputs = keras.Input(shape=(input_dim,))

x = layers.Dense(width_first, activation="relu", kernel_regularizer=keras.regularizers.l2(l2))(encoder_inputs)
x = layers.Dropout(dropout)(x)

x = layers.Dense(width_first // 2, activation="relu", kernel_regularizer=keras.regularizers.l2(l2))(x)
x = layers.Dropout(dropout)(x)

#x = layers.Dense(width_first // 4, activation="relu", kernel_regularizer=keras.regularizers.l2(l2))(x)
#x = layers.Dropout(dropout)(x)

#x = layers.Dense(width_first // 8, activation="relu", kernel_regularizer=keras.regularizers.l2(l2))(x)
#x = layers.Dropout(dropout)(x)

z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])

encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")
encoder.summary()


In [ ]:
# Build the decoder

In [ ]:
latent_inputs = keras.Input(shape=(latent_dim,))

#x = layers.Dense(width_first // 8, activation="relu")(latent_inputs)
#x = layers.Dropout(dropout)(x)
#x = layers.Dense(width_first // 4, activation="relu")(x)
#x = layers.Dropout(dropout)(x)
x = layers.Dense(width_first // 2, activation="relu")(latent_inputs)
x = layers.Dropout(dropout)(x)
x = layers.Dense(width_first, activation="relu")(x)
x = layers.Dropout(dropout)(x)
decoder_outputs = layers.Dense(input_dim, activation="linear")(x)

decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")
decoder.summary()


In [ ]:
# Define the VAE as a Model with a custom train_step

In [ ]:
"""β-VAE: loss = MSE + β·KL."""
class VAE(keras.Model):
    def __init__(self, encoder, decoder, beta=beta_kl, **kw):
        super().__init__(**kw)
        self.encoder     = encoder
        self.decoder     = decoder
        self.beta        = beta
        self.loss_tracker= keras.metrics.Mean(name="loss")
        self.mse_tracker = keras.metrics.Mean(name="mse")
        self.kl_tracker  = keras.metrics.Mean(name="kl")

    @property
    def metrics(self):          # <– Keras będzie to resetował co epokę
        return [self.loss_tracker, self.mse_tracker, self.kl_tracker]

    def call(self, x):
        _, _, z = self.encoder(x)
        return self.decoder(z)

    def train_step(self, x):
        with tf.GradientTape() as tape:
            mu, logvar, z = self.encoder(x, training=True)
            #logvar = tf.clip_by_value(logvar, -3.0, 3.0)       BEZ TEGO
            recon = self.decoder(z, training=True)
            mse = tf.reduce_mean(tf.square(x - recon), axis=1)
            kl  = -0.5 * tf.reduce_sum(1 + logvar - tf.square(mu) - tf.exp(logvar), axis=1)
            #loss = tf.reduce_mean(mse + self.beta * kl)
            loss = tf.reduce_mean(mse + self.beta * kl)

        grads = tape.gradient(loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.loss_tracker.update_state(loss)
        self.mse_tracker.update_state(tf.reduce_mean(mse))
        self.kl_tracker.update_state(tf.reduce_mean(kl))

        return {
            "loss": self.loss_tracker.result(),
            "mse":  self.mse_tracker.result(),
            "kl":   self.kl_tracker.result()
        }

    # val step (żeby działał val_loss)
    def test_step(self, x):
        mu, logvar, z = self.encoder(x, training=False)
        #logvar = tf.clip_by_value(logvar, -3.0, 3.0)
        recon  = self.decoder(z, training=False)
        mse    = tf.reduce_mean(tf.square(x - recon), axis=1)
        kl     = -0.5 * tf.reduce_sum(1 + logvar - tf.square(mu) - tf.exp(logvar), axis=1)
        loss   = tf.reduce_mean(mse + self.beta * kl)
        self.loss_tracker.update_state(loss)
        self.mse_tracker.update_state(tf.reduce_mean(mse))
        self.kl_tracker.update_state(tf.reduce_mean(kl))
        return {
            "loss": self.loss_tracker.result(),
            "mse":  self.mse_tracker.result(),
            "kl":   self.kl_tracker.result()
        }




In [ ]:
# Train the VAE

In [ ]:
vae = VAE(encoder, decoder, beta=beta_kl)
vae.compile(optimizer=keras.optimizers.Adam(1e-4, clipnorm=1.0),
           metrics=["mse", "kl"])

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True
)

history = vae.fit(
    X_train,
    epochs=100,
    batch_size=32,
    shuffle=True,
    validation_data=(X_test, None),    
    callbacks=[early_stop],
)


# --- 2. Inicjalizacja słownika wyników tylko raz ---
results = {}


# --- 5. RMSE rekonstrukcji po inverse_transform ---
def global_rmse(X1, X2):
    return np.sqrt(np.mean((X1 - X2) ** 2))

X_train_orig = scaler.inverse_transform(X_train).reshape(-1, 100, 55, 3)
X_test_orig  = scaler.inverse_transform(X_test).reshape(-1, 100, 55, 3)
X_anom_orig  = scaler.inverse_transform(X_anom).reshape(-1, 100, 55, 3)

X_train_rec = scaler.inverse_transform(vae.predict(X_train)).reshape(-1, 100, 55, 3)
X_test_rec  = scaler.inverse_transform(vae.predict(X_test)).reshape(-1, 100, 55, 3)
X_anom_rec  = scaler.inverse_transform(vae.predict(X_anom)).reshape(-1, 100, 55, 3)

results["train_Rmse"] = global_rmse(X_train_orig, X_train_rec)
results["val_Rmse"]   = global_rmse(X_test_orig,  X_test_rec)
results["anom_Rmse"]  = global_rmse(X_anom_orig,  X_anom_rec)

# --- 6. Wyniki detekcji anomalii ---
eval_results = evaluate(vae, X_train, X_test, X_anom)
results.update(eval_results)

# --- 7. KL-divergence i MSE anomalii w Z-score ---
results["kl_divergence"] = float(history.history["kl"][-1])
results["train_kl"]      = float(history.history["kl"][-1])
results["val_kl"]        = float(history.history["val_kl"][-1])

# Dodanie anom_mse (Z-score)
anom_recon = vae.predict(X_anom, verbose=0)
results["anom_mse"] = float(np.mean((X_anom - anom_recon) ** 2))

# --- 8. Hiperparametry i metadane treningu ---
results.update({
    "latent_dim":    latent_dim,
    "depth":         depth,
    "width_first":   width_first,
    "batch_size":    batch_size,
    "learning_rate": learning_rate,
    "beta(KL)":      beta_kl,
    "l2":            l2,
    "dropout":       dropout,
    "epochs":        len(history.history["loss"]),
    "stopped_epoch": early_stop.stopped_epoch,
    "params":        vae.count_params(),

})


log_run(**results)

import matplotlib.pyplot as plt

def plot_training_history(history_dict):
    plt.figure(figsize=(10, 5))

    # Rekonstrukcja (MSE)
    if "mse" in history_dict:
        plt.plot(history_dict["mse"], label="Train MSE")
    if "val_mse" in history_dict:
        plt.plot(history_dict["val_mse"], label="Val MSE")

    # KL-divergence (tylko dla VAE)
    if "kl" in history_dict:
        plt.plot(history_dict["kl"], label="Train KL")
    if "val_kl" in history_dict:
        plt.plot(history_dict["val_kl"], label="Val KL")

    plt.xlabel("Epoch")
    plt.ylabel("Loss / KL / MSE")
    plt.title("Training History")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


plot_training_history(history.history)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =============================
# Konfiguracja
# =============================
latent_dim   = 4
num_samples  = 10
sigmas       = [0.5, 1.0, 2.0, 10.0]   # możesz zmienić 10 -> 4 lub 8
seed         = 0
np.random.seed(seed)

wanted = [
    "LFHD","RFHD","LBHD","RBHD","GLAB","CLAV","STRN","C7","T2","T10","RBAK",
    "LMAS","RMAS","LASI","LPSI","LTHI","LTHAP","LTHAD","LKNE","LTIB",
    "LTIAP","LTIAD","LANK","LHEE","LTOE","LFMH","LVMH","RASI","RPSI",
    "RTHI","RTHAP","RTHAD","RKNE","RTIB","RTIAP","RTIAD","RANK","RHEE",
    "RTOE","RFMH","RVMH","LSHO","LUPA","LELB","LFRM","LWRA","LWRB","LFIN",
    "RSHO","RUPA","RELB","RFRM","RWRA","RWRB","RFIN"
]
LHEE_idx = wanted.index("LHEE")

# Ustal mapowanie osi (DOPASUJ do Twoich danych!)
# Jeśli u Ciebie "wysokość" jest w osi 2, to poniżej podpisz to jako "vertical"
AX_LABEL = {0: "Z", 1: "X", 2: "Y"}  # klasycznie; jeśli masz inaczej, zmień nazwy
axes_to_show = [0, 1, 2]

# =============================
# Dane referencyjne (walk + anom)
# =============================
walk_seq = scaler.inverse_transform(X_test[0].reshape(1, -1))[0].reshape(100, 55, 3)
anom_seq = scaler.inverse_transform(X_anom[50].reshape(1, -1))[0].reshape(100, 55, 3)

# (opcjonalnie) rekonstrukcja dla anomalii
# recon = vae.predict(X_anom[0:1], verbose=0)
# recon_seq = scaler.inverse_transform(recon)[0].reshape(100, 55, 3)

# =============================
# Funkcje pomocnicze
# =============================
def sample_decode(num_samples, sigma):
    """Zwraca gen_seq_batch o kształcie (num_samples, 100, 55, 3) w jednostkach oryginalnych."""
    z = np.random.normal(0.0, sigma, size=(num_samples, latent_dim))
    gen_scaled = vae.decoder.predict(z, verbose=0)              # (num_samples, 16500)
    gen_orig   = scaler.inverse_transform(gen_scaled)           # (num_samples, 16500)
    gen_seq    = gen_orig.reshape(num_samples, 100, 55, 3)
    return gen_seq

def amplitude_p95_p5(trajs):
    """trajs: (num_samples, 100) -> średnia amplituda (p95 - p5) po próbkach"""
    amp = np.percentile(trajs, 95, axis=1) - np.percentile(trajs, 5, axis=1)
    return float(np.mean(amp))

def mse_to_ref(trajs, ref):
    """trajs: (num_samples, 100), ref: (100,)"""
    return float(np.mean((trajs - ref[None, :])**2))

# =============================
# Rysunek: 3 wiersze (osie) x 4 kolumny (sigma)
# =============================
fig, axes = plt.subplots(
    nrows=len(axes_to_show),
    ncols=len(sigmas),
    figsize=(4.2*len(sigmas), 2.8*len(axes_to_show)),
    dpi=200,
    sharex=True
)

for col, sigma in enumerate(sigmas):
    gen_seq_batch = sample_decode(num_samples, sigma)  # (num_samples,100,55,3)

    for row, ax_id in enumerate(axes_to_show):
        ax = axes[row, col] if len(axes_to_show) > 1 else axes[col]

        # referencje
        walk_ref = walk_seq[:, LHEE_idx, ax_id]
        anom_ref = anom_seq[:, LHEE_idx, ax_id]

        # generacje
        gen_trajs = gen_seq_batch[:, :, LHEE_idx, ax_id]  # (num_samples,100)

        # cienkie generacje
        for i in range(num_samples):
            ax.plot(gen_trajs[i], color="green", alpha=0.5, linewidth=1.0, label="generated" if (i == 0 and row == 0 and col == 0) else None)
        
            # grube referencje
            ax.plot(walk_ref, color="blue", linewidth=2.5, label="walk" if (i == 0 and row==0 and col==0) else None)
            ax.plot(anom_ref, color="red",  linewidth=2.5, label="run" if (i == 0 and row==0 and col==0) else None)
            
            # metryki do obrony (krótko w rogu)
            #amp = amplitude_p95_p5(gen_trajs)
            #mse = mse_to_ref(gen_trajs, walk_ref)
            #ax.text(
            #    0.02, 0.95,
            #    f"amp~{amp:.1f}\nMSE~{mse:.1f}",
            #    transform=ax.transAxes,
            #    va="top",
            #    fontsize=8
            #)
    
            if row == 0:
                ax.set_title(f"$\\sigma$ = {sigma}", fontsize=11)
    
            if col == 0:
                ax.set_ylabel(f"LHEE {AX_LABEL[ax_id]}", fontsize=10)
    
            ax.grid(True, alpha=0.3)

# wspólne opisy
for ax in axes[-1, :]:
    ax.set_xlabel("frame (0–99)", fontsize=10)

handles, labels = axes[0,0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", fontsize=9, frameon=True)

plt.tight_layout()
plt.savefig("LHEE_xyz_sampling_sigma_grid.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: LHEE_xyz_sampling_sigma_grid.png")


In [ ]:
import pandas as pd

df = pd.read_csv("runs.csv")          # wczytuje zaktualizowany plik
best = df.sort_values("val_mse").head(5)   # 5 najlepszych
display(best)


In [ ]:
from eval_core import evaluate

results = evaluate(vae, X_train, X_test, X_anom)

# RMSE po odwrotnej transformacji Z-score
def global_rmse(X1, X2):
    return np.sqrt(np.mean((X1 - X2)**2))

X_train_orig = scaler.inverse_transform(X_train).reshape(-1, 100, 55, 3)
X_test_orig  = scaler.inverse_transform(X_test).reshape(-1, 100, 55, 3)
X_anom_orig  = scaler.inverse_transform(X_anom).reshape(-1, 100, 55, 3)

X_train_rec = scaler.inverse_transform(vae.predict(X_train)).reshape(-1, 100, 55, 3)
X_test_rec  = scaler.inverse_transform(vae.predict(X_test)).reshape(-1, 100, 55, 3)
X_anom_rec  = scaler.inverse_transform(vae.predict(X_anom)).reshape(-1, 100, 55, 3)

results["RMSE_train"] = global_rmse(X_train_orig, X_train_rec)
results["RMSE_val"]   = global_rmse(X_test_orig, X_test_rec)
results["RMSE_anom"]  = global_rmse(X_anom_orig, X_anom_rec)

from pprint import pprint
print(results)


In [ ]:
# import numpy as np
import matplotlib.pyplot as plt

# === Ustawienia ===
latent_dim = 4  
num_samples = 10          
sigma_gen = 1           # sigma przy losowaniu z N(0, sigma_gen^2)

# Lista markerów i indeks LHEE (jak wcześniej)
wanted = [
    "LFHD","RFHD","LBHD","RBHD","GLAB","CLAV","STRN","C7","T2","T10","RBAK",
    "LMAS","RMAS","LASI","LPSI","LTHI","LTHAP","LTHAD","LKNE","LTIB",
    "LTIAP","LTIAD","LANK","LHEE","LTOE","LFMH","LVMH","RASI","RPSI",
    "RTHI","RTHAP","RTHAD","RKNE","RTIB","RTIAP","RTIAD","RANK","RHEE",
    "RTOE","RFMH","RVMH","LSHO","LUPA","LELB","LFRM","LWRA","LWRB","LFIN",
    "RSHO","RUPA","RELB","RFRM","RWRA","RWRB","RFIN"
]
LHEE_idx = wanted.index("LHEE")

# Pobieramy oryginalne sekwencje do porównania:
walk_seq = scaler.inverse_transform(X_test[0].reshape(1, -1))[0].reshape(100, 55, 3)
anom_seq = scaler.inverse_transform(X_anom[0].reshape(1, -1))[0].reshape(100, 55, 3)

recon = vae.predict(X_anom[0:1], verbose=0)  # (1, 16500) rekonstukcja
recon_seq = scaler.inverse_transform(recon)[0].reshape(100, 55, 3)

# === Losujemy 100 wektorów z ~ N(0, sigma_gen^2) ===
z_batch = np.random.normal(loc=0.0, scale=sigma_gen, size=(num_samples, latent_dim))

# === Dekodujemy wszystkie naraz (jeśli pamięć pozwala) ===
#    Ważne: vae.decoder.predict przyjmie (100, latent_dim) i zwróci (100, 16500)
gen_scaled_batch = vae.decoder.predict(z_batch, verbose=0)  # shape = (100, 16500)

# === Inwersja Z-score i reshape do (100, 100, 55, 3) ===
#    Najpierw do (100, 16500), potem do (100, 100, 55, 3)
gen_orig_batch = scaler.inverse_transform(gen_scaled_batch)  # shape = (100, 16500)
gen_seq_batch = gen_orig_batch.reshape(num_samples, 100, 55, 3)  # (100, 100, 55, 3)

# === Rysowanie ===
plt.figure(figsize=(12, 6))

# 0 - Z, 1 - X , 2-Y ASIS
axis = 2
# 1) Rysujemy oryginalny chód i anomalię jako grubsze linie
plt.plot(walk_seq[:, LHEE_idx, axis], color='blue', linewidth=3, label="chód (oryginalny)")
plt.plot(anom_seq[:, LHEE_idx, axis], color='red', linewidth=3, label="bieg (anomalia, oryginalny)")

# 2) Rysujemy 100 wygenerowanych trajektorii LHEE_Z jako cienkie, półprzezroczyste linie
for i in range(num_samples):
    traj = gen_seq_batch[i, :, LHEE_idx, axis]   # (100,)
    plt.plot(traj, color='green', alpha=0.3, linewidth=1)

#rekonstrukcja    
#plt.plot(recon_seq[:, LHEE_idx, 2], color='purple', linewidth=2, linestyle='--', label="rekonstrukcja")
# 3) Dodajemy opisy osi i legendę
plt.title(f"{num_samples} wygenerowanych trajektorii LHEE_Y (sigma={sigma_gen}) vs. oryginał i anomalia(oryginał biegu)", fontsize=14)
plt.xlabel("klatka", fontsize=12)
plt.ylabel("wysokość [mm]", fontsize=12)
plt.xticks([0, 25, 50, 75, 100])
plt.yticks([0, 100, 200, 300, 400, 500])
plt.legend(loc="upper right", fontsize=10)
plt.grid(True)
plt.tight_layout()
plt.show()


mu, logvar, _ = vae.encoder(X_train[0:1])
print("z_mean:", mu.numpy())
print("z_logvar:", logvar.numpy())


In [ ]:
z_mean, z_logvar, _ = encoder.predict(X_test)
z_std = np.exp(0.5 * z_logvar)

print("Średnie μ (po wymiarach):", np.round(np.mean(z_mean, axis=0), 3))
print("Odchylenie μ (std)      :", np.round(np.std(z_mean,  axis=0), 3))
print("Średnie σ (po wymiarach):", np.round(np.mean(z_std,  axis=0), 3))


In [ ]:
recon_test  = vae.predict(X_test,  verbose=0)
errors_test = np.mean((X_test  - recon_test)**2, axis=1)

recon_anom  = vae.predict(X_anom,  verbose=0)
errors_anom = np.mean((X_anom  - recon_anom)**2, axis=1)

from sklearn.metrics import roc_auc_score
y_true  = np.concatenate([np.zeros(len(errors_test)),  np.ones(len(errors_anom))])
y_score = np.concatenate([errors_test, errors_anom])
auc     = roc_auc_score(y_true, y_score)
print("ROC-AUC detekcji anomalii:", auc)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ——————————————————————————————————————
# 1) Przygotowanie danych
# ——————————————————————————————————————
# X_train, X_test: już spłaszczone (N,16500) i z-score'd
# scaler: StandardScaler, fit na X_train

# wczytanie i transformacja anomalii
anom = np.load("E:/marcel.furs/Documents/Data_Run_Walk/dataset_anomaly_multi_fixed_OSTATECZNY.npy")
anom = anom.reshape(-1, 100*55*3)
anom = scaler.transform(anom)

# ——————————————————————————————————————
# 2) Funkcje obliczające MSE i KL per-próbka
# ——————————————————————————————————————
def mse_samples(X, model, batch_size=64):
    rec = model.predict(X, batch_size=batch_size, verbose=0)
    return np.mean((X - rec)**2, axis=1)

def kl_samples(X, encoder, batch_size=64):
    mu, logvar, _ = encoder.predict(X, batch_size=batch_size, verbose=0)
    # KL = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    return -0.5 * np.sum(1 + logvar - mu**2 - np.exp(logvar), axis=1)

# ——————————————————————————————————————
# 3) Policzenie metryk
# ——————————————————————————————————————
mse_tr = mse_samples(X_train, vae)
mse_te = mse_samples(X_test,  vae)
mse_an = mse_samples(anom,     vae)

kl_tr  = kl_samples(X_train, encoder)
kl_te  = kl_samples(X_test,  encoder)
kl_an  = kl_samples(anom,     encoder)

# próg detekcji na 3σ (z train)
thr_mse = mse_tr.mean() + 3*mse_tr.std()
thr_kl  = kl_tr.mean()  + 3*kl_tr.std()
print("Shapes:", X_train.shape, X_test.shape, X_anom.shape)

In [ ]:
# ——————————————————————————————————————
# 4) Histogramy MSE i KL
# ——————————————————————————————————————
fig, axes = plt.subplots(1,2, figsize=(12,4))

# MSE
axes[0].hist(mse_tr, bins=40, alpha=0.6, label="train")
axes[0].hist(mse_te, bins=40, alpha=0.6, label="test")
axes[0].hist(mse_an, bins=40, alpha=0.6, label="anomalia")
axes[0].axvline(thr_mse, color='k', ls='--', label="thr = μ+3σ")
axes[0].set_title("Błąd rekonstrukcji (MSE)")
axes[0].set_xlabel("MSE"); axes[0].set_ylabel("Liczba próbek")
axes[0].legend()

# KL
axes[1].hist(kl_tr, bins=40, alpha=0.6, label="train")
axes[1].hist(kl_te, bins=40, alpha=0.6, label="test")
axes[1].hist(kl_an, bins=40, alpha=0.6, label="anomalia")
axes[1].axvline(thr_kl, color='k', ls='--', label="thr = μ+3σ")
axes[1].set_title("Rozkład KL-divergence")
axes[1].set_xlabel("KL"); axes[1].set_ylabel("Liczba próbek")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# lista markerów
wanted = [
    "LFHD","RFHD","LBHD","RBHD","GLAB","CLAV","STRN","C7","T2","T10","RBAK",
    "LMAS","RMAS","LASI","LPSI","LTHI","LTHAP","LTHAD","LKNE","LTIB",
    "LTIAP","LTIAD","LANK","LHEE","LTOE","LFMH","LVMH","RASI","RPSI",
    "RTHI","RTHAP","RTHAD","RKNE","RTIB","RTIAP","RTIAD","RANK","RHEE",
    "RTOE","RFMH","RVMH","LSHO","LUPA","LELB","LFRM","LWRA","LWRB","LFIN",
    "RSHO","RUPA","RELB","RFRM","RWRA","RWRB","RFIN"
]

# indeks markera LHEE
LHEE_idx = wanted.index("LHEE")

# --- wybór sekwencji ---
walk_flat = X_test[430]        # sekwencja chodu (po normalizacji)
anom_flat = X_anom[100]        # sekwencja biegu (po normalizacji)
#z_rand = np.random.normal(0, 1, size=(1, latent_dim))
#gen_flat = vae.decoder.predict(z_rand)[0]  # shape = (16500,)
# --- rekonstrukcja ---
walk_rec_flat = vae.predict(walk_flat.reshape(1, -1), verbose=0)[0]
anom_rec_flat = vae.predict(anom_flat.reshape(1, -1), verbose=0)[0]

# --- odwrócenie Z-score i reshape ---
walk_orig = scaler.inverse_transform(walk_flat.reshape(1, -1))[0].reshape(100, 55, 3)
walk_rec  = scaler.inverse_transform(walk_rec_flat.reshape(1, -1))[0].reshape(100, 55, 3)

anom_orig = scaler.inverse_transform(anom_flat.reshape(1, -1))[0].reshape(100, 55, 3)
anom_rec  = scaler.inverse_transform(anom_rec_flat.reshape(1, -1))[0].reshape(100, 55, 3)

# --- rysowanie ---
plt.figure(figsize=(12, 6))
plt.plot(walk_orig[:, LHEE_idx, 2], lw=2, label="chód (oryginał)", color="tab:blue")
plt.plot(walk_rec[:, LHEE_idx, 2],  lw=2, label="chód (rekonstrukcja)", color="tab:cyan", linestyle="--")

plt.plot(anom_orig[:, LHEE_idx, 2], lw=2, label="bieg (oryginał)", color="tab:red")
plt.plot(anom_rec[:, LHEE_idx, 2],  lw=2, label="bieg (rekonstrukcja)", color="tab:pink", linestyle="--")

plt.title("Porównanie trajektorii markera LHEE w osi Y")
plt.xlabel("klatka")
plt.ylabel("wysokość [mm]")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# 1) Pobierz z_mean z enkodera (pierwszy zwracany tensor)
z_mean_walk, _, _ = encoder.predict(X_test,  batch_size=256)
z_mean_run,  _, _ = encoder.predict(X_anom, batch_size=256)

# 2) Złącz i przygotuj etykiety
X_lat = np.vstack([z_mean_walk, z_mean_run])
labels = np.array([0]*len(z_mean_walk) + [1]*len(z_mean_run))

# 3) t-SNE 3D
X_3d = TSNE(n_components=3, random_state=42).fit_transform(X_lat)

# 4) Rysunek
fig = plt.figure(figsize=(7,6))
ax = fig.add_subplot(projection="3d")
for lab, color, name in zip([0,1], ["green","red"], ["walk","run"]):
    idx = labels == lab
    ax.scatter(
        X_3d[idx,0], X_3d[idx,1], X_3d[idx,2],
        c=color, s=10, alpha=0.7, label=name
    )
ax.set_title("VAE latent (z_mean) – t-SNE 3D")
ax.set_xlabel("t-SNE-1")
ax.set_ylabel("t-SNE-2")
ax.set_zlabel("t-SNE-3")
ax.legend()
plt.tight_layout()
plt.show()
print("Shapes:", X_train.shape, X_test.shape, X_anom.shape)
# — opcjonalnie: zapisz współrzędne do pliku —
# np.save("vae_latent_tsne3d.npy", X_3d)
print("z_mean_walk:", z_mean_walk.shape)
print("z_mean_run :", z_mean_run.shape)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# 1) Pobierz z_mean z enkodera
z_mean_walk, _, _ = encoder.predict(X_test,  batch_size=256)
z_mean_run,  _, _ = encoder.predict(X_anom, batch_size=256)

# 2) Złącz dane i etykiety
X_lat = np.vstack([z_mean_walk, z_mean_run])
labels = np.array([0]*len(z_mean_walk) + [1]*len(z_mean_run))

# 3) t-SNE 2D
X_2d = TSNE(n_components=2, random_state=42).fit_transform(X_lat)

# 4) Wykres 2D
plt.figure(figsize=(6,6))
for lab, color, name in zip([0,1], ["green","red"], ["walk","run"]):
    idx = labels == lab
    plt.scatter(
        X_2d[idx,0], X_2d[idx,1],
        c=color, s=20, alpha=0.7, label=name
    )
plt.title("VAE latent (z_mean) – t-SNE 2D")
plt.xlabel("t-SNE-1")
plt.ylabel("t-SNE-2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
#===============================

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
# Wczytaj oryginalną sekwencję
orig_data = np.load("E:/marcel.furs/Documents/Data_Run_Walk/NajnowszedoMgr/dataset_anomaly_multi_fixed_OSTATECZNY.npy")  # lub dataset_test
orig_data1 = np.load("E:/marcel.furs/Documents/Data_Run_Walk/NajnowszedoMgr/dataset_train_multi_fixed_OSTATECZNY.npy") 
orig_seq = orig_data[0]  # np. pierwsza sekwencja

# === Lista markerów i krawędzi w postaci indeksów ===
wanted = [ "LFHD","RFHD","LBHD","RBHD","GLAB","CLAV","STRN","C7","T2","T10","RBAK",
           "LMAS","RMAS","LASI","LPSI","LTHI","LTHAP","LTHAD","LKNE","LTIB",
           "LTIAP","LTIAD","LANK","LHEE","LTOE","LFMH","LVMH","RASI","RPSI",
           "RTHI","RTHAP","RTHAD","RKNE","RTIB","RTIAP","RTIAD","RANK","RHEE",
           "RTOE","RFMH","RVMH","LSHO","LUPA","LELB","LFRM","LWRA","LWRB","LFIN",
           "RSHO","RUPA","RELB","RFRM","RWRA","RWRB","RFIN" ]

#edges = [
#    ("LASI","LPSI"),("LASI","RASI"),("RASI","RPSI"),("LPSI","RPSI"),
#    ("LASI","LKNE"),("LKNE","LANK"),("LANK","LTOE"),("LANK","LHEE"),
#    ("RASI","RKNE"),("RKNE","RANK"),("RANK","RTOE"),("RANK","RHEE"),
#    ("CLAV","C7"),("C7","STRN"),
#    ("CLAV","LSHO"),("LSHO","LELB"),("LELB","LWRA"),("LWRA","LFIN"),
#    ("CLAV","RSHO"),("RSHO","RELB"),("RELB","RWRA"),("RWRA","RFIN"),
#    ("RFHD","LFHD"),("LBHD","RBHD"),("RFHD","RBHD"),("LFHD","LBHD"),
#    ("RFHD","GLAB"),("LFHD","GLAB")
#]
edges = [
    ("LPSI", "RPSI"),  # MIEDNICA
    ("C7", "T2"), ("T2", "T10"), ("T10", "RPSI"), ("T10", "LPSI"), ("RSHO", "T2"), ("LSHO", "T2"), # KRĘGOSŁUP I DO BARK
    ("LSHO", "LUPA"), ("LUPA", "LELB"), ("LELB", "LFRM"), ("LFRM", "LFIN"), # LEWA REKA
    ("RSHO", "RUPA"), ("RUPA", "RELB"), ("RELB", "RFRM"), ("RFRM", "RFIN"), # PRAWA REKA
    ("LPSI", "LTHI"), ("LTHI", "LKNE"), ("LKNE", "LTIB"), ("LTIB", "LANK"), ("LANK", "LHEE"), ("LHEE", "LVMH"), # LEWA NOGA
    ("RPSI", "RTHI"), ("RTHI", "RKNE"), ("RKNE", "RTIB"), ("RTIB", "RANK"), ("RANK", "RHEE"), ("RHEE", "RVMH"), # PRAWA NOGA
    ("RMAS", "RBHD"), ("RBHD", "GLAB"),
    ("C7", "RMAS") # GŁOWA Z KREGOSLUPEM
]
edge_idx = [(wanted.index(a), wanted.index(b)) for a,b in edges]

# === 1) Wygeneruj nową sztuczną sekwencję ===
latent_dim = 4  # jak w moimm modelu

range1 = 1
z_rand = np.random.normal(loc=0, scale=range1, size=(1, latent_dim))
print(z_rand)
recons = decoder.predict(z_rand)
gen_flat = recons[0]

# === 2) Odwrócenie z-score + reshape ===
gen_flat_orig = scaler.inverse_transform(gen_flat.reshape(1, -1))[0]
reco_seq = gen_flat_orig.reshape(100, 55, 3)

# -----------------------------------------------------------------
# === 3) Centrowanie na miednicy (oddzielnie dla obu sekwencji) ===
LASI, RASI = wanted.index("LASI"), wanted.index("RASI")

pelvis_orig = (orig_seq[0, LASI] + orig_seq[0, RASI]) / 2
orig_c      = orig_seq - pelvis_orig          # oryginał w swoim układzie

pelvis_reco = (reco_seq[0, LASI] + reco_seq[0, RASI]) / 2
reco_c      = reco_seq - pelvis_reco          # rekonstrukcja w swoim układzie

# === Obliczenia dodatkowych metryk ===
def get_height(seq, top_idx, bottom_idx):
    # Wysokość = różnica Y pomiędzy głową a kostkami (np. LFHD a LANK)
    return np.mean(seq[:, top_idx, 2] - seq[:, bottom_idx, 2])

def get_shoulder_width(seq, lsho_idx, rsho_idx):
    # Szerokość barków (średnia odległość w osi X/Z, cała sekwencja)
    return np.mean(np.linalg.norm(seq[:, lsho_idx, [0,2]] - seq[:, rsho_idx, [0,2]], axis=1))

# Indeksy
LFHD = wanted.index("LFHD")
LANK = wanted.index("LANK")
LSHO = wanted.index("LSHO")
RSHO = wanted.index("RSHO")

# Oryginał
orig_height = get_height(orig_seq, LFHD, LANK)
orig_width  = get_shoulder_width(orig_seq, LSHO, RSHO)

# Generowany
reco_height = get_height(reco_seq, LFHD, LANK)
reco_width  = get_shoulder_width(reco_seq, LSHO, RSHO)

print(f"Oryginał:    Wzrost ~ {orig_height:.1f} mm, szerokość barków ~ {orig_width:.1f} mm")
print(f"Wygenerowany: Wzrost ~ {reco_height:.1f} mm, szerokość barków ~ {reco_width:.1f} mm")
# === 4) Przygotowanie wykresu ===
fig = plt.figure(figsize=(10,5))
ax1 = fig.add_subplot(121, projection='3d')
ax2 = fig.add_subplot(122, projection='3d')

# ustalenie jednakowych limitów i widoku
xyz_min = np.minimum(orig_c.min((0,1)), reco_c.min((0,1)))
xyz_max = np.maximum(orig_c.max((0,1)), reco_c.max((0,1)))
mid     = (xyz_max + xyz_min) / 2
rng     = (xyz_max - xyz_min).max() / 2
for ax in (ax1, ax2):
    ax.set_xlim(mid[0]-rng, mid[0]+rng)
    ax.set_ylim(mid[1]-rng, mid[1]+rng)
    ax.set_zlim(mid[2]-rng, mid[2]+rng)
    ax.set_box_aspect([1, 1, 1])                                                          #TU ZMIANA
    ax.view_init(elev=15, azim=60)

#ax1.set_title("Oryginał")
#ax2.set_title("Wygenerowana sekwencja VAE")

# linie i punkty
lines1 = [ax1.plot([],[],[],'k-', lw=2)[0] for _ in edge_idx]
pts1   = ax1.scatter([],[],[], c='b', s=10)
lines2 = [ax2.plot([],[],[],'k-', lw=2)[0] for _ in edge_idx]
pts2   = ax2.scatter([],[],[], c='r', s=10)

# funkcje init/update
def init():
    for ln in lines1+lines2:
        ln.set_data([], []); ln.set_3d_properties([])
    pts1._offsets3d = ([],[],[])
    pts2._offsets3d = ([],[],[])
    return lines1 + [pts1] + lines2 + [pts2]

def update(frame):
    p1 = orig_c[frame]
    p2 = reco_c[frame]
    for ln,(i,j) in zip(lines1, edge_idx):
        ln.set_data(p1[[i,j], 0], p1[[i,j], 1])
        ln.set_3d_properties(p1[[i,j], 2])
    pts1._offsets3d = (p1[:,0], p1[:,1], p1[:,2])
    for ln,(i,j) in zip(lines2, edge_idx):
        ln.set_data(p2[[i,j], 0], p2[[i,j], 1])
        ln.set_3d_properties(p2[[i,j], 2])
    pts2._offsets3d = (p2[:,0], p2[:,1], p2[:,2])
    return lines1 + [pts1] + lines2 + [pts2]

# === 5) Generowanie GIF i zapis ===
ani = animation.FuncAnimation(
    fig, update, frames=range(0, 100), init_func=init, interval=33, blit=True
)

ani.save(" 21TEST_KL.gif", writer="pillow", fps=30)
fig.text(0.5, 0.02, f'Oryginał: {orig_height:.1f} mm, szerokość barków: {orig_width:.1f} mm\n'
                    f'Wygenerowany: {reco_height:.1f} mm, szerokość barków: {reco_width:.1f} mm',
         ha='center', fontsize=10)
plt.close(fig)

print(" 21TEST_KL.gif")
# srednia osob to 174cm
# roznica 23cm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# ==========================================================
# 0) Wczytaj dane
# ==========================================================
orig_data = np.load(r"E:/marcel.furs/Documents/Data_Run_Walk/NajnowszedoMgr/dataset_test_multi_fixed_OSTATECZNY.npy")
orig_seq = orig_data[0]  # (100,55,3)

# ==========================================================
# 1) Markery + krawędzie
# ==========================================================
wanted = [ "LFHD","RFHD","LBHD","RBHD","GLAB","CLAV","STRN","C7","T2","T10","RBAK",
           "LMAS","RMAS","LASI","LPSI","LTHI","LTHAP","LTHAD","LKNE","LTIB",
           "LTIAP","LTIAD","LANK","LHEE","LTOE","LFMH","LVMH","RASI","RPSI",
           "RTHI","RTHAP","RTHAD","RKNE","RTIB","RTIAP","RTIAD","RANK","RHEE",
           "RTOE","RFMH","RVMH","LSHO","LUPA","LELB","LFRM","LWRA","LWRB","LFIN",
           "RSHO","RUPA","RELB","RFRM","RWRA","RWRB","RFIN" ]

edges = [
    ("LPSI", "RPSI"),
    ("C7", "T2"), ("T2", "T10"), ("T10", "RPSI"), ("T10", "LPSI"), ("RSHO", "T2"), ("LSHO", "T2"),
    ("LSHO", "LUPA"), ("LUPA", "LELB"), ("LELB", "LFRM"), ("LFRM", "LFIN"),
    ("RSHO", "RUPA"), ("RUPA", "RELB"), ("RELB", "RFRM"), ("RFRM", "RFIN"),
    ("LPSI", "LTHI"), ("LTHI", "LKNE"), ("LKNE", "LTIB"), ("LTIB", "LANK"), ("LANK", "LHEE"), ("LHEE", "LVMH"),
    ("RPSI", "RTHI"), ("RTHI", "RKNE"), ("RKNE", "RTIB"), ("RTIB", "RANK"), ("RANK", "RHEE"), ("RHEE", "RVMH"),
    ("RMAS", "RBHD"), ("RBHD", "GLAB"),
    ("C7", "RMAS")
]
edge_idx = [(wanted.index(a), wanted.index(b)) for a, b in edges]

# ==========================================================
# 2) Generacja VAE (zakładam: decoder, scaler)
# ==========================================================
latent_dim = 4
z_rand = np.random.normal(0, 1.0, size=(1, latent_dim))
recons = decoder.predict(z_rand, verbose=0)
gen_flat = recons[0]
gen_flat_orig = scaler.inverse_transform(gen_flat.reshape(1, -1))[0]
reco_seq = gen_flat_orig.reshape(100, 55, 3)

# ==========================================================
# 3) Centrowanie na miednicy PER FRAME (ważne!)
# ==========================================================
LASI_idx, RASI_idx = wanted.index("LASI"), wanted.index("RASI")
pelvis_orig_t = 0.5 * (orig_seq[:, LASI_idx, :] + orig_seq[:, RASI_idx, :])
pelvis_reco_t = 0.5 * (reco_seq[:, LASI_idx, :] + reco_seq[:, RASI_idx, :])
orig_c = orig_seq - pelvis_orig_t[:, None, :]
reco_c = reco_seq - pelvis_reco_t[:, None, :]

# ==========================================================
# 4) Limity osi wspólne
# ==========================================================
xyz_min = np.minimum(orig_c.min((0,1)), reco_c.min((0,1)))
xyz_max = np.maximum(orig_c.max((0,1)), reco_c.max((0,1)))
mid     = (xyz_max + xyz_min) / 2
rng     = (xyz_max - xyz_min).max() / 2

def setup_ax(ax, elev=15, azim=120):
    ax.set_xlim(mid[0]-rng, mid[0]+rng)
    ax.set_ylim(mid[1]-rng, mid[1]+rng)
    ax.set_zlim(mid[2]-rng, mid[2]+rng)
    ax.set_box_aspect([1, 1, 1])
    ax.view_init(elev=elev, azim=azim)
    ax.set_axis_off()

def draw_stroboscopic(ax, seq_c, frames, shift_x=0.0, alpha_min=0.10, alpha_max=0.55, lw=1.2):
    # rysujemy wiele klatek na JEDNEJ osi, z rosnącą alfą
    alphas = np.linspace(alpha_min, alpha_max, len(frames))
    for k, (f, a) in enumerate(zip(frames, alphas)):
        p = seq_c[f].copy()
        p[:, 0] += k * shift_x  # opcjonalnie: mały shift, żeby ruch był czytelniejszy
        for (i, j) in edge_idx:
            ax.plot([p[i,0], p[j,0]],
                    [p[i,1], p[j,1]],
                    [p[i,2], p[j,2]],
                    color="black", lw=lw, alpha=a)

# ==========================================================
# 5) STROBOSKOP: wybór klatek
# ==========================================================
N_SNAP = 12
frames = np.linspace(0, 99, N_SNAP, dtype=int)

fig = plt.figure(figsize=(10, 4.5))
ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122, projection="3d")

setup_ax(ax1, elev=0, azim=0)
setup_ax(ax2, elev=0, azim=0)

# shift_x: daj np. 15–30 mm jeśli chcesz lekkie „rozciągnięcie” ruchu
shift_x = 1.5 * (xyz_max[0] - xyz_min[0]) / (len(frames)-1)
draw_stroboscopic(ax1, orig_c, frames, shift_x=shift_x)
draw_stroboscopic(ax2, reco_c, frames, shift_x=shift_x)


ax1.set_title("Original", fontsize=10, pad=2)
ax2.set_title("Generated", fontsize=10, pad=2)

plt.tight_layout()
plt.savefig("vae_stroboscopic.png", dpi=300, bbox_inches="tight")
plt.close()
print("✔ Zapisano: vae_stroboscopic.png")

# ==========================================================
VERT = 2
marker_to_idx = {name: i for i, name in enumerate(wanted)}

HEAD_IDXS = [marker_to_idx[m] for m in ["LFHD","RFHD","LBHD","RBHD"]]
LHEE = marker_to_idx["LHEE"]
RHEE = marker_to_idx["RHEE"]
LSHO = marker_to_idx["LSHO"]
RSHO = marker_to_idx["RSHO"]
LASI = marker_to_idx["LASI"]
RASI = marker_to_idx["RASI"]

def height_proxy(seq, p_head=95, p_foot=5):
    head_y = seq[:, HEAD_IDXS, VERT].mean(axis=1)  # (T,)
    foot_y = np.minimum(seq[:, LHEE, VERT], seq[:, RHEE, VERT])
    return float(np.percentile(head_y, p_head) - np.percentile(foot_y, p_foot))

def shoulder_width(seq):
    d = seq[:, LSHO, :] - seq[:, RSHO, :]
    return float(np.linalg.norm(d, axis=1).mean())

def lhee_amp_y(seq):
    y = seq[:, LHEE, VERT]
    return float(np.percentile(y, 95) - np.percentile(y, 5))

def pelvis_drift(seq):
    pelvis = 0.5*(seq[:, LASI, :] + seq[:, RASI, :])  # (T,3)
    return float(np.linalg.norm(pelvis[-1] - pelvis[0]))

# --- policz
h_o = height_proxy(orig_seq)
h_g = height_proxy(reco_seq)

w_o = shoulder_width(orig_seq)
w_g = shoulder_width(reco_seq)

a_o = lhee_amp_y(orig_seq)
a_g = lhee_amp_y(reco_seq)

d_o = pelvis_drift(orig_seq)
d_g = pelvis_drift(reco_seq)

print("=== Single-sequence metrics (Original vs Generated) ===")
print(f"Height-proxy [mm]:   orig={h_o:.1f}, gen={h_g:.1f}, Δ={abs(h_g-h_o):.1f}")
print(f"Shoulder width [mm]: orig={w_o:.1f}, gen={w_g:.1f}, Δ={abs(w_g-w_o):.1f}")
print(f"LHEE amp Y [mm]:     orig={a_o:.1f}, gen={a_g:.1f}, Δ={abs(a_g-a_o):.1f}")
print(f"Pelvis drift [mm]:   orig={d_o:.1f}, gen={d_g:.1f}, Δ={abs(d_g-d_o):.1f}")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# ==========================================================
# 0) Wczytaj dane
# ==========================================================
orig_data = np.load(r"E:/marcel.furs/Documents/Data_Run_Walk/NajnowszedoMgr/dataset_test_multi_fixed_OSTATECZNY.npy")
orig_seq = orig_data[0]  # (100,55,3)

# ==========================================================
# 1) Markery + krawędzie
# ==========================================================
wanted = [ "LFHD","RFHD","LBHD","RBHD","GLAB","CLAV","STRN","C7","T2","T10","RBAK",
           "LMAS","RMAS","LASI","LPSI","LTHI","LTHAP","LTHAD","LKNE","LTIB",
           "LTIAP","LTIAD","LANK","LHEE","LTOE","LFMH","LVMH","RASI","RPSI",
           "RTHI","RTHAP","RTHAD","RKNE","RTIB","RTIAP","RTIAD","RANK","RHEE",
           "RTOE","RFMH","RVMH","LSHO","LUPA","LELB","LFRM","LWRA","LWRB","LFIN",
           "RSHO","RUPA","RELB","RFRM","RWRA","RWRB","RFIN" ]

edges = [
    ("LPSI", "RPSI"), # MIEDNICA
    ("C7", "T2"), ("T2", "T10"), ("T10", "RPSI"), ("T10", "LPSI"), ("RSHO", "T2"), ("LSHO", "T2"), # KRĘGOSŁUP I DO BARK
    ("LSHO", "LUPA"), ("LUPA", "LELB"), ("LELB", "LFRM"), ("LFRM", "LFIN"), # LEWA REKA
    ("RSHO", "RUPA"), ("RUPA", "RELB"), ("RELB", "RFRM"), ("RFRM", "RFIN"), # PRAWA REKA
    ("LPSI", "LTHI"), ("LTHI", "LKNE"), ("LKNE", "LTIB"), ("LTIB", "LANK"), ("LANK", "LHEE"), ("LHEE", "LVMH"), # LEWA NOGA
    ("RPSI", "RTHI"), ("RTHI", "RKNE"), ("RKNE", "RTIB"), ("RTIB", "RANK"), ("RANK", "RHEE"), ("RHEE", "RVMH"), # PRAWA NOGA
    ("RMAS", "RBHD"), ("RBHD", "GLAB"),
    ("C7", "RMAS") # GŁOWA Z KREGOSLUPEM
]
edge_idx = [(wanted.index(a), wanted.index(b)) for a, b in edges]
edge_points = sorted(set(i for e in edge_idx for i in e))
# ==========================================================
# 2) Generacja VAE (zakładam: decoder, scaler)
# ==========================================================
latent_dim = 4
range1 = 1.0
z_rand = np.random.normal(loc=0, scale=range1, size=(1, latent_dim))
print("z_rand:", z_rand)

recons = decoder.predict(z_rand, verbose=0)
gen_flat = recons[0]

gen_flat_orig = scaler.inverse_transform(gen_flat.reshape(1, -1))[0]
reco_seq = gen_flat_orig.reshape(100, 55, 3)

# ==========================================================
# 3) Centrowanie na miednicy PER FRAME (ważne!)
# ==========================================================
LASI_idx, RASI_idx = wanted.index("LASI"), wanted.index("RASI")

pelvis_orig_t = 0.5 * (orig_seq[:, LASI_idx, :] + orig_seq[:, RASI_idx, :])  # (100,3)
pelvis_reco_t = 0.5 * (reco_seq[:, LASI_idx, :] + reco_seq[:, RASI_idx, :])

orig_c = orig_seq - pelvis_orig_t[:, None, :]
reco_c = reco_seq - pelvis_reco_t[:, None, :]

# ==========================================================
# 4) Wspólne limity osi (jak w GIF)
# ==========================================================
xyz_min = np.minimum(orig_c.min((0,1)), reco_c.min((0,1)))
xyz_max = np.maximum(orig_c.max((0,1)), reco_c.max((0,1)))
mid     = (xyz_max + xyz_min) / 2
rng     = (xyz_max - xyz_min).max() / 2

def setup_ax(ax):
    ax.set_xlim(mid[0]-rng, mid[0]+rng)
    ax.set_ylim(mid[1]-rng, mid[1]+rng)
    ax.set_zlim(mid[2]-rng, mid[2]+rng)
    ax.set_box_aspect([1, 1, 1])
    ax.view_init(elev=0, azim=0)  # identycznie jak GIF
    ax.set_axis_off()
    ax.margins(0)

def draw_skeleton(ax, p, pts_color):
    # p: (55,3)
    for (i, j) in edge_idx:
        ax.plot([p[i,0], p[j,0]],
                [p[i,1], p[j,1]],
                [p[i,2], p[j,2]],
                color="black", lw=1.6)
    ax.scatter(
    p[edge_points, 0],
    p[edge_points, 1],
    p[edge_points, 2],
    c=pts_color,
    s=1
)

# ==========================================================
# 5) SIATKA UJĘĆ (10 klatek) – jak klatki z GIFa
# ==========================================================
N_FRAMES = 10
frames = np.linspace(0, 99, N_FRAMES, dtype=int)

fig = plt.figure(figsize=(1.5*N_FRAMES, 6.2))

# --- Original: rząd 1 ---
for col, f in enumerate(frames):
    ax = fig.add_subplot(2, N_FRAMES, col + 1, projection="3d")
    setup_ax(ax)
    draw_skeleton(ax, orig_c[f], pts_color="blue")
    ax.set_title(f"{int(100*f/99)}%", fontsize=7, pad=2)

# --- VAE: rząd 2 ---
for col, f in enumerate(frames):
    ax = fig.add_subplot(2, N_FRAMES, N_FRAMES + col + 1, projection="3d")
    setup_ax(ax)
    draw_skeleton(ax, reco_c[f], pts_color="red")
    ax.set_title(f"{int(100*f/99)}%", fontsize=7, pad=2)

# Etykiety wierszy
fig.text(0.006, 0.72, "Original", rotation=90, va="center", fontsize=12)
fig.text(0.006, 0.28, "Generated ", rotation=90, va="center", fontsize=12)

#plt.tight_layout()
plt.subplots_adjust(
    left=0.02,
    right=0.99,
    top=0.95,
    bottom=0.05,
    wspace=0, 
    hspace=0.05
)

plt.savefig("vae_frames_like_gif.png", dpi=300)
plt.close()

print("✔ Zapisano: vae_frames_like_gif.png")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# 0) Wczytaj dane
# ==========================================================
orig_data = np.load(r"E:/marcel.furs/Documents/Data_Run_Walk/NajnowszedoMgr/dataset_test_multi_fixed_OSTATECZNY.npy")
orig_seq = orig_data[0]  # (100,55,3)

# ==========================================================
# 1) Markery + krawędzie
# ==========================================================
wanted = [ "LFHD","RFHD","LBHD","RBHD","GLAB","CLAV","STRN","C7","T2","T10","RBAK",
           "LMAS","RMAS","LASI","LPSI","LTHI","LTHAP","LTHAD","LKNE","LTIB",
           "LTIAP","LTIAD","LANK","LHEE","LTOE","LFMH","LVMH","RASI","RPSI",
           "RTHI","RTHAP","RTHAD","RKNE","RTIB","RTIAP","RTIAD","RANK","RHEE",
           "RTOE","RFMH","RVMH","LSHO","LUPA","LELB","LFRM","LWRA","LWRB","LFIN",
           "RSHO","RUPA","RELB","RFRM","RWRA","RWRB","RFIN" ]

edges = [
    ("LPSI", "RPSI"),
    ("C7", "T2"), ("T2", "T10"), ("T10", "RPSI"), ("T10", "LPSI"), ("RSHO", "T2"), ("LSHO", "T2"),
    ("LSHO", "LUPA"), ("LUPA", "LELB"), ("LELB", "LFRM"), ("LFRM", "LFIN"),
    ("RSHO", "RUPA"), ("RUPA", "RELB"), ("RELB", "RFRM"), ("RFRM", "RFIN"),
    ("LPSI", "LTHI"), ("LTHI", "LKNE"), ("LKNE", "LTIB"), ("LTIB", "LANK"), ("LANK", "LHEE"), ("LHEE", "LVMH"),
    ("RPSI", "RTHI"), ("RTHI", "RKNE"), ("RKNE", "RTIB"), ("RTIB", "RANK"), ("RANK", "RHEE"), ("RHEE", "RVMH"),
    ("C7","RMAS")
]
edge_idx = [(wanted.index(a), wanted.index(b)) for a, b in edges]

LASI_idx, RASI_idx = wanted.index("LASI"), wanted.index("RASI")

# ==========================================================
# 2) Generacja VAE (zakładam: decoder, scaler)
# ==========================================================
latent_dim = 4
range1 = 1.0
z_rand = np.random.normal(loc=0, scale=range1, size=(1, latent_dim))
recons = decoder.predict(z_rand, verbose=0)
gen_flat = recons[0]
gen_flat_orig = scaler.inverse_transform(gen_flat.reshape(1, -1))[0]
reco_seq = gen_flat_orig.reshape(100, 55, 3)

# ==========================================================
# 3) Kierunek marszu z miednicy + obrót układu (żeby iść w prawo)
# ==========================================================
def pelvis_t(seq):
    return 0.5 * (seq[:, LASI_idx, :] + seq[:, RASI_idx, :])  # (T,3)

pel_o = pelvis_t(orig_seq)
pel_r = pelvis_t(reco_seq)

# Ustalamy "forward" z oryginału: XY (poziom), od początku do końca
# (to działa dobrze nawet jak jest trochę szumu)
v = (pel_o[-1, :2] - pel_o[0, :2]).astype(float)
if np.linalg.norm(v) < 1e-6:
    # awaryjnie: PCA na trajektorii
    traj = pel_o[:, :2] - pel_o[:, :2].mean(axis=0, keepdims=True)
    cov = traj.T @ traj
    vals, vecs = np.linalg.eigh(cov)
    v = vecs[:, np.argmax(vals)]
v = v / (np.linalg.norm(v) + 1e-12)

# rotation: obracamy tak, żeby v -> [1,0]
# jeśli v = [vx, vy], to cos = vx, sin = vy
vx, vy = v[0], v[1]
R2 = np.array([[ vx,  vy],
               [-vy,  vx]], dtype=float)  # to jest obrót o -atan2(vy,vx)

def align_seq(seq, pel_ref):
    """1) przenosi do układu z początkiem w miednicy (z 1. klatki),
       2) obraca w poziomie tak, aby kierunek marszu był w prawo."""
    out = seq.copy()
    out = out - pel_ref[0, None, None, :]          # odejmij miednicę z 1. klatki (NIE per-frame!)
    xy = out[:, :, :2].reshape(-1, 2)              # (T*J,2)
    xy_rot = (xy @ R2.T).reshape(out.shape[0], out.shape[1], 2)
    out[:, :, :2] = xy_rot
    return out

orig_a = align_seq(orig_seq, pel_o)
reco_a = align_seq(reco_seq, pel_r)

# ==========================================================
# 4) Rzut 2D: (forward = X po obrocie) vs vertical (Z)
# ==========================================================
VERT_AXIS = 2  # u Ciebie wygląda, że Z to góra/dół
def to_2d(seq):
    x = seq[:, :, 0]            # forward
    y = seq[:, :, VERT_AXIS]    # vertical
    return x, y

orig_x, orig_y = to_2d(orig_a)
reco_x, reco_y = to_2d(reco_a)

# ==========================================================
# 5) Stroboskop: przesuwamy kolejne klatki w prawo (jak Fig. c)
# ==========================================================
N_SNAP = 18
frames = np.linspace(0, 99, N_SNAP, dtype=int)

# shift w prawo (w jednostkach danych, mm) — dobierz pod gust:
# jako stabilny wybór: 0.12 * "szerokość" sylwetki w forward
width_ref = (np.percentile(orig_x, 95) - np.percentile(orig_x, 5))
SHIFT_X = 0.4 * width_ref  # np. 12% szerokości na klatkę

ALPHA = 0.25
LW = 1.4

def draw_stroboscopic(ax, X, Y, title):
    # limity: na podstawie pierwszej klatki i shiftów
    x_min = np.min(X[frames, :]) + 0 * SHIFT_X
    x_max = np.max(X[frames, :]) + (len(frames)-1) * SHIFT_X
    y_min = np.min(Y[frames, :])
    y_max = np.max(Y[frames, :])

    ax.set_xlim(x_min - 0.05*(x_max-x_min), x_max + 0.05*(x_max-x_min))
    ax.set_ylim(y_min - 0.05*(y_max-y_min), y_max + 0.05*(y_max-y_min))
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, fontsize=12)

    for k, f in enumerate(frames):
        dx = k * SHIFT_X
        for (i, j) in edge_idx:
            ax.plot([X[f, i] + dx, X[f, j] + dx],
                    [Y[f, i],       Y[f, j]],
                    color="black", lw=1.6, alpha=0.45)

# ==========================================================
# 6) Rysunek 2-panelowy
# ==========================================================
fig, axes = plt.subplots(2, 1, figsize=(12, 4), dpi=200)

draw_stroboscopic(axes[0], orig_x, orig_y, "Original")
draw_stroboscopic(axes[1], reco_x, reco_y, "Generated")

plt.tight_layout()
plt.savefig("paper_stroboscopic.png", dpi=300, bbox_inches="tight")
plt.close()
print(" Zapisano: paper_stroboscopic.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# =============================
# Konfiguracja markerów
# =============================
wanted = [
    "LFHD","RFHD","LBHD","RBHD","GLAB","CLAV","STRN","C7","T2","T10","RBAK",
    "LMAS","RMAS","LASI","LPSI","LTHI","LTHAP","LTHAD","LKNE","LTIB",
    "LTIAP","LTIAD","LANK","LHEE","LTOE","LFMH","LVMH","RASI","RPSI",
    "RTHI","RTHAP","RTHAD","RKNE","RTIB","RTIAP","RTIAD","RANK","RHEE",
    "RTOE","RFMH","RVMH","LSHO","LUPA","LELB","LFRM","LWRA","LWRB","LFIN",
    "RSHO","RUPA","RELB","RFRM","RWRA","RWRB","RFIN"
]

LFHD_idx = wanted.index("LFHD")
LHEE_idx = wanted.index("LHEE")
LSHO_idx = wanted.index("LSHO")
RSHO_idx = wanted.index("RSHO")
LASI_idx = wanted.index("LASI")
RASI_idx = wanted.index("RASI")

# =============================
# Funkcje do obliczania metryk
# =============================
def compute_height_proxy(seq):
    """Wysokość: głowa - pięta (oś Z)"""
    head_z = seq[:, LFHD_idx, 2]
    heel_z = seq[:, LHEE_idx, 2]
    return np.mean(head_z - heel_z)

def compute_shoulder_width(seq):
    """Szerokość barków: odległość LSHO-RSHO"""
    lsho = seq[:, LSHO_idx, :]
    rsho = seq[:, RSHO_idx, :]
    dist = np.linalg.norm(lsho - rsho, axis=1)
    return np.mean(dist)

def compute_stride_length(seq):
    """Długość kroku: przemieszczenie LHEE w osi X"""
    heel_x = seq[:, LHEE_idx, 1]
    return abs(heel_x[-1] - heel_x[0])

def compute_foot_clearance(seq):
    """Wysokość podniesienia stopy: zakres LHEE w osi Z"""
    heel_z = seq[:, LHEE_idx, 2]
    return np.max(heel_z) - np.min(heel_z)

def compute_pelvis_sway(seq):
    """Zakres ruchu miednicy w osi bocznej (Y)"""
    pelvis_y = (seq[:, LASI_idx, 0] + seq[:, RASI_idx, 0]) / 2
    return np.max(pelvis_y) - np.min(pelvis_y)

def compute_smoothness(seq):
    """Płynność: średnia zmiana pozycji między klatkami"""
    diffs = np.diff(seq, axis=0)
    return np.mean(np.abs(diffs))

def compute_all_metrics(seq):
    """Oblicza wszystkie metryki dla jednej sekwencji"""
    return {
        'height': compute_height_proxy(seq),
        'shoulder': compute_shoulder_width(seq),
        'stride': compute_stride_length(seq),
        'clearance': compute_foot_clearance(seq),
        'pelvis': compute_pelvis_sway(seq),
        'smoothness': compute_smoothness(seq)
    }

# =============================
# Dane rzeczywiste
# =============================
X_all = np.concatenate([X_train], axis=0)
n_real = X_all.shape[0]

real_metrics = {k: [] for k in ['height', 'shoulder', 'stride', 'clearance', 'pelvis', 'smoothness']}

for i in range(n_real):
    seq = scaler.inverse_transform(X_all[i].reshape(1, -1))[0].reshape(100, 55, 3)
    m = compute_all_metrics(seq)
    for k, v in m.items():
        real_metrics[k].append(v)

# =============================
# Dane wygenerowane
# =============================
n_gen = 30
sigma = 1.0
z = np.random.normal(0.0, sigma, size=(n_gen, latent_dim))
gen_scaled = vae.decoder.predict(z, verbose=0)

gen_metrics = {k: [] for k in ['height', 'shoulder', 'stride', 'clearance', 'pelvis', 'smoothness']}

for i in range(n_gen):
    seq = scaler.inverse_transform(gen_scaled[i].reshape(1, -1))[0].reshape(100, 55, 3)
    m = compute_all_metrics(seq)
    for k, v in m.items():
        gen_metrics[k].append(v)

# =============================
# Budowanie tabeli
# =============================
metric_names = {
    'height': 'Height-proxy [mm]',
    'shoulder': 'Shoulder width [mm]',
    'stride': 'Stride length [mm]',
    'clearance': 'Foot clearance [mm]',
    'pelvis': 'Pelvis sway [mm]',
}

rows = []
for key, name in metric_names.items():
    real_vals = real_metrics[key]
    gen_vals = gen_metrics[key]
    
    real_mean, real_std = np.mean(real_vals), np.std(real_vals)
    gen_mean, gen_std = np.mean(gen_vals), np.std(gen_vals)
    
    t_stat, p_val = stats.ttest_ind(real_vals, gen_vals)
    
    rows.append({
        "Metric": name,
        f"Real (n={n_real})": f"{real_mean:.1f} ± {real_std:.1f}",
        f"Generated (n={n_gen}, σ={sigma})": f"{gen_mean:.1f} ± {gen_std:.1f}",
        "Δmean": f"{abs(gen_mean - real_mean):.1f}",
    })

df_metrics = pd.DataFrame(rows)
display(df_metrics)

# =============================
# Zapis PNG
# =============================
def save_df_as_png(df, title, out_png, dpi=200):
    fig, ax = plt.subplots(figsize=(12, 1 + 0.45 * len(df)))
    ax.axis("off")
    ax.set_title(title, fontsize=12, pad=10)
    tbl = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        loc="center"
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.4)
    plt.tight_layout()
    plt.savefig(out_png, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_png}")

save_df_as_png(df_metrics, "Comparison of gait metrics: Real vs VAE-generated sequences", "table_gait_metrics.png")